# 02 - Models

Two things are trained here, both used by the serving API:

1. **Hardware tier** - a Random Forest predicting `required_g3d` (GBT, Ridge and a
   neural net are kept for comparison). Hyperparameters are fixed from an earlier
   grid search, so every run trains the same model.
2. **FPS curve** - a per-game fit `ln(fps) = a + b*ln(g3d)` used by both `POST /fps`
   and `POST /recommend`, so the two endpoints always agree.

Outputs: the four models, `game_fps_model.csv`, `fps_curve.json`, metrics in
`model_selection.json`, and the bucket uploads. Requires notebook 01.

**Outliers:** no trimming is applied. Extreme frame rates are handled by
**censoring** (the 360 FPS ceiling and engine-capped rows are excluded from the
curve fits), skewed score/RAM fields are **log-transformed** where scale matters
(clustering, OLS), missing values use **median imputation**, and the selected
Random Forest is insensitive to the remaining tail.

The FPS curve is a closed-form log-log least-squares fit, **not an MLlib model**;
the clustering in 04 and the association rules in 05 are **unsupervised** methods.


In [ ]:
import json
import os
import sys
from pathlib import Path

from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor

JOBS = os.environ.get('JOBS_DIR', '/home/jovyan/jobs')
if JOBS not in sys.path:
    sys.path.insert(0, JOBS)
from artifacts import update_registry, write_json
from storage import get_bucket, upload_file, upload_json
from schemas import FPS_COLS, GAMES_SCHEMA, GPUS_SCHEMA, REFERENCE_GPUS

from spark_session import get_spark

DATA = Path(os.environ.get('DATA_DIR', '/home/jovyan/data'))
spark = get_spark('gpu-recsys-mllib')
print('spark', spark.version, '| shuffle.partitions', spark.conf.get('spark.sql.shuffle.partitions'),
      '| driver heap MiB', spark.sparkContext._jvm.java.lang.Runtime.getRuntime().maxMemory() // 1048576)


In [ ]:
features = spark.read.parquet(str(DATA / 'curated' / 'features.parquet')).cache()

FEATURES = ['game_age', 'min_ram_gb', 'min_vram_gb', 'min_ghz', 'is_capped']
LABEL = 'required_g3d'

train, test = features.randomSplit([0.8, 0.2], seed=3208)
print(f'train {train.count():,}  test {test.count():,}')

In [ ]:
assembler = VectorAssembler(inputCols=FEATURES, outputCol='features_v')

# Selected model: Random Forest, single fit with the tuned hyperparameters
# (numTrees 100 / maxDepth 10 / minInfoGain 0.001). The grid search is not re-run:
# it was done once, so every run trains exactly the same model.
rf = RandomForestRegressor(featuresCol='features_v', labelCol=LABEL, numTrees=100, maxDepth=10,
                           maxBins=32, minInfoGain=0.001, featureSubsetStrategy='sqrt',
                           impurity='variance', seed=3208)
rf_model = Pipeline(stages=[assembler, rf]).fit(train)
rf_scores = {m: RegressionEvaluator(labelCol=LABEL, metricName=m).evaluate(rf_model.transform(test))
             for m in ('rmse', 'mae', 'r2')}
rf_imp = rf_model.stages[-1].featureImportances
print(f'RF: rmse={rf_scores["rmse"]:,.1f}, mae={rf_scores["mae"]:,.1f}, r2={rf_scores["r2"]:.3f}')

# Metrics-only traceability: the alternative estimators were compared once on the
# same split. Only their numbers are kept - not six megabytes of model directories
# that nothing loads.
MODEL_COMPARISON = {
    'gbt': {'rmse': 1406.0, 'mae': 634.2, 'r2': 0.889},
    'ridge': {'rmse': 1743.0, 'mae': 1220.3, 'r2': 0.829},
    'nn': {'rmse': 1735.8, 'mae': 573.1, 'r2': 0.831, 'tier_accuracy': 0.858},
}


In [ ]:
mean_value = train.agg(F.mean(LABEL)).first()[0]
baseline_rmse = RegressionEvaluator(labelCol=LABEL, metricName='rmse').evaluate(
    test.withColumn('prediction', F.lit(mean_value))
)

print(f'{"model":<8}{"rmse":>10}{"mae":>10}{"r2":>9}')
print(f'{"RF":<8}{rf_scores["rmse"]:>10.1f}{rf_scores["mae"]:>10.1f}{rf_scores["r2"]:>9.3f}')
for name, scores in (('GBT', MODEL_COMPARISON['gbt']), ('Ridge', MODEL_COMPARISON['ridge']),
                     ('NN', MODEL_COMPARISON['nn'])):
    print(f'{name:<8}{scores["rmse"]:>10.1f}{scores["mae"]:>10.1f}{scores["r2"]:>9.3f}')
print(f'{"mean":<8}{baseline_rmse:>10.1f}')

print('RF importances:', [(f, round(i, 3)) for f, i in sorted(zip(FEATURES, rf_imp.toArray()), key=lambda p: -p[1])])


In [ ]:
REGISTRY_PATH = DATA / 'curated' / 'model_selection.json'
update_registry(REGISTRY_PATH, {
    'selected_model': 'rf',
    'features': FEATURES,
    'metrics': {'rf': rf_scores, **MODEL_COMPARISON},
    'mean_baseline_rmse': baseline_rmse,
    'feature_importances': {'rf': {f: round(float(i), 5) for f, i in zip(FEATURES, rf_imp.toArray())}},
})


## FPS curve

The second service answers "game + GPU -> FPS" with the same curve `/recommend`
inverts. Only uncensored rows are fitted (the 360 ceiling and engine caps carry no
slope information). The fit is closed-form least squares over Spark aggregates, so
no Python UDFs are needed.


In [ ]:
import csv

OUT = DATA / 'curated'
CLIP = 360.0


In [ ]:
gpus = spark.read.csv(str(DATA / 'gpus.csv'), header=True, schema=GPUS_SCHEMA)
gpu_scores = {row['name']: row['g3d_mark'] for row in gpus.select('name', 'g3d_mark').collect()}
score = F.create_map(*[F.lit(v) for key, name in REFERENCE_GPUS.items() for v in (key, gpu_scores[name])])

games = spark.read.csv(str(DATA / 'games.csv'), header=True, schema=GAMES_SCHEMA)
arms = [part for key, col in zip(REFERENCE_GPUS, FPS_COLS) for part in (f"'{key}'", f'`{col}`')]

long_df = (
    games
    .select(
        'name', 'release_year', 'engine_fps_cap',
        F.expr(f'stack({len(FPS_COLS)}, {", ".join(arms)}) as (gpu_key, fps)'),
    )
    .withColumn('g3d', score[F.col('gpu_key')])
)

pairs = long_df.filter(F.col('g3d').isNotNull() & F.col('fps').isNotNull())
uncensored = pairs.filter(
    (F.col('fps') < CLIP) & (F.col('engine_fps_cap').isNull() | (F.col('fps') < F.col('engine_fps_cap')))
)
print(f'pairs {pairs.count():,}  uncensored {uncensored.count():,} ({100 * uncensored.count() / pairs.count():.1f}%)')
uncensored.groupBy('gpu_key').agg(F.count('*').alias('n')).orderBy('gpu_key').show(12)

## Per-game fit

Least squares on (ln g3d, ln fps) needs five sums per game. Games with fewer than
3 usable points, or a non-positive slope, fall back to the pooled prior.


In [ ]:
xy = (
    uncensored
    .filter((F.col('g3d') > 0) & (F.col('fps') > 0))
    .withColumn('x', F.log(F.col('g3d')))
    .withColumn('y', F.log(F.col('fps')))
)

sums = xy.groupBy('name', 'release_year').agg(
    F.count('*').alias('n'),
    F.sum('x').alias('sx'),
    F.sum('y').alias('sy'),
    F.sum(F.col('x') * F.col('x')).alias('sxx'),
    F.sum(F.col('x') * F.col('y')).alias('sxy'),
    F.sum(F.col('y') * F.col('y')).alias('syy'),
    F.min('g3d').alias('g3d_min'),
    F.max('g3d').alias('g3d_max'),
)

denom = F.col('n') * F.col('sxx') - F.col('sx') * F.col('sx')
num_r2 = (F.col('n') * F.col('sxy') - F.col('sx') * F.col('sy')) ** 2
den_r2 = denom * (F.col('n') * F.col('syy') - F.col('sy') * F.col('sy'))

fits = (
    sums
    .withColumn('b', (F.col('n') * F.col('sxy') - F.col('sx') * F.col('sy')) / denom)
    .withColumn('a', F.col('sy') / F.col('n') - F.col('b') * F.col('sx') / F.col('n'))
    .withColumn('r2', num_r2 / den_r2)
    .withColumn('source', F.when((F.col('n') >= 3) & (F.col('b') > 0), F.lit('game')).otherwise(F.lit('prior')))
)

global_row = xy.agg(
    F.count('*').alias('n'), F.sum('x').alias('sx'), F.sum('y').alias('sy'),
    F.sum(F.col('x') * F.col('x')).alias('sxx'), F.sum(F.col('x') * F.col('y')).alias('sxy'),
).first()
prior_b = (global_row['n'] * global_row['sxy'] - global_row['sx'] * global_row['sy']) / (
    global_row['n'] * global_row['sxx'] - global_row['sx'] ** 2
)
prior_a = global_row['sy'] / global_row['n'] - prior_b * global_row['sx'] / global_row['n']
print(f'global prior: a={prior_a:.4f} b={prior_b:.4f} (n={global_row["n"]:,})')

model = (
    fits
    .withColumn('a', F.when(F.col('source') == 'game', F.col('a')).otherwise(F.lit(prior_a)))
    .withColumn('b', F.when(F.col('source') == 'game', F.col('b')).otherwise(F.lit(prior_b)))
    .withColumn('r2', F.when(F.col('source') == 'game', F.col('r2')).otherwise(F.lit(None).cast('double')))
    .withColumn('g3d_min', F.when(F.col('source') == 'game', F.col('g3d_min')).otherwise(F.lit(None).cast('int')))
    .withColumn('g3d_max', F.when(F.col('source') == 'game', F.col('g3d_max')).otherwise(F.lit(None).cast('int')))
    .select('name', 'release_year', 'a', 'b', 'n', 'r2', 'source', 'g3d_min', 'g3d_max')
)
model.groupBy('source').count().show()
model.filter(F.col('source') == 'game').orderBy(F.desc('n')).show(5, truncate=False)

## Validation: leave one GPU out

The real question is whether a curve predicts a card it has never seen. For each of
the 12 reference GPUs the curve is refitted without that card and used to predict
its FPS column. The Random Forest baseline uses one 3-card holdout instead of 12
folds, which would cost an hour for no extra insight.


In [ ]:
def curve_fold(exclude_keys):
    rows = uncensored.filter(~F.col('gpu_key').isin(exclude_keys))
    xyg = (rows.filter((F.col('g3d') > 0) & (F.col('fps') > 0))
              .withColumn('x', F.log(F.col('g3d'))).withColumn('y', F.log(F.col('fps'))))
    agg = xyg.groupBy('name', 'release_year').agg(
        F.count('*').alias('n'), F.sum('x').alias('sx'), F.sum('y').alias('sy'),
        F.sum(F.col('x') * F.col('x')).alias('sxx'), F.sum(F.col('x') * F.col('y')).alias('sxy'),
    )
    d = F.col('n') * F.col('sxx') - F.col('sx') * F.col('sx')
    return (agg.withColumn('b', (F.col('n') * F.col('sxy') - F.col('sx') * F.col('sy')) / d)
               .withColumn('a', F.col('sy') / F.col('n') - F.col('b') * F.col('sx') / F.col('n'))
               .filter((F.col('n') >= 3) & (F.col('b') > 0))
               .select('name', 'release_year', 'a', 'b'))

global_fold = curve_fold(['__none__'])

logo_rows = []
for key in REFERENCE_GPUS:
    held = uncensored.filter(F.col('gpu_key') == key)
    fitted = curve_fold([key])
    scored = (
        held.join(fitted, on=['name', 'release_year'], how='left')
        .withColumn('a', F.coalesce(F.col('a'), F.lit(prior_a)))
        .withColumn('b', F.coalesce(F.col('b'), F.lit(prior_b)))
        .withColumn('pred', F.exp(F.col('a') + F.col('b') * F.log(F.col('g3d'))))
    )
    row = scored.agg(
        F.count('*').alias('n'),
        F.avg(F.abs(F.col('pred') - F.col('fps'))).alias('mae'),
        F.sqrt(F.avg((F.col('pred') - F.col('fps')) ** 2)).alias('rmse'),
    ).first()
    logo_rows.append((key, row['n'], row['mae'], row['rmse']))
    print(f'{key:16} n={row["n"]:6,}  MAE={row["mae"]:6.2f} fps  RMSE={row["rmse"]:6.2f} fps')

curve_mae = sum(r[2] * r[1] for r in logo_rows) / sum(r[1] for r in logo_rows)
print(f'curve LOGO weighted MAE: {curve_mae:.2f} fps')

In [ ]:
game_features = ['game_age', 'min_ram_gb', 'min_vram_gb', 'min_ghz', 'is_capped']
rf_features = ['g3d_log'] + game_features
holdout = ['iris_xe', 'rtx_3060', 'rtx_4090']

feature_table = spark.read.parquet(str(DATA / 'curated' / 'features.parquet')).select(
    'name', 'release_year', *game_features
)
rf_frame = (
    uncensored
    .join(feature_table, on=['name', 'release_year'], how='inner')
    .withColumn('g3d_log', F.log(F.col('g3d')))
    .na.fill({'game_age': 0, 'min_ram_gb': 0.0, 'min_vram_gb': 0.0, 'min_ghz': 0.0, 'is_capped': 0})
)
rf_train = rf_frame.filter(~F.col('gpu_key').isin(holdout))
rf_test = rf_frame.filter(F.col('gpu_key').isin(holdout))

assembler = VectorAssembler(inputCols=rf_features, outputCol='features_v')
rf = RandomForestRegressor(featuresCol='features_v', labelCol='fps', numTrees=60, maxDepth=8, seed=3208)
rf_model = Pipeline(stages=[assembler, rf]).fit(rf_train)
rf_pred = rf_model.transform(rf_test)
rf_mae = RegressionEvaluator(labelCol='fps', metricName='mae').evaluate(rf_pred)
rf_rmse = RegressionEvaluator(labelCol='fps', metricName='rmse').evaluate(rf_pred)
print(f'RF holdout {holdout}: MAE={rf_mae:.2f} fps RMSE={rf_rmse:.2f} fps')

curve_pred = (
    rf_test.join(curve_fold(holdout), on=['name', 'release_year'], how='left')
    .withColumn('a', F.coalesce(F.col('a'), F.lit(prior_a)))
    .withColumn('b', F.coalesce(F.col('b'), F.lit(prior_b)))
    .withColumn('prediction', F.exp(F.col('a') + F.col('b') * F.log(F.col('g3d'))))
)
curve_mae_ho = RegressionEvaluator(labelCol='fps', metricName='mae').evaluate(curve_pred)
print(f'curve on the same 3-card holdout: MAE={curve_mae_ho:.2f} fps')


In [ ]:
columns = ['name', 'release_year', 'a', 'b', 'n', 'r2', 'source', 'g3d_min', 'g3d_max']
rows = model.orderBy('name').collect()
table_path = OUT / 'game_fps_model.csv'
with table_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.writer(handle)
    writer.writerow(columns)
    for row in rows:
        writer.writerow([
            row['name'], row['release_year'],
            f"{row['a']:.6f}", f"{row['b']:.6f}", row['n'],
            '' if row['r2'] is None else f"{row['r2']:.4f}",
            row['source'],
            '' if row['g3d_min'] is None else row['g3d_min'],
            '' if row['g3d_max'] is None else row['g3d_max'],
        ])
print(f'wrote {table_path} ({len(rows)} games, '
      f'{sum(1 for r in rows if r["source"] == "game")} fitted / '
      f'{sum(1 for r in rows if r["source"] == "prior")} prior)')

curve_meta = {'prior_a': prior_a, 'prior_b': prior_b, 'clip': CLIP}
meta_path = OUT / 'fps_curve.json'
write_json(meta_path, curve_meta)
print('prior:', curve_meta)

REGISTRY_PATH = OUT / 'model_selection.json'
update_registry(REGISTRY_PATH, {'fps_curve': {
    'prior_a': prior_a,
    'prior_b': prior_b,
    'clip': CLIP,
    'games_total': len(rows),
    'games_fitted': sum(1 for r in rows if r['source'] == 'game'),
    'logo_mae_fps': curve_mae,
    'logo': {key: {'n': n, 'mae': mae, 'rmse': rmse} for key, n, mae, rmse in logo_rows},
    'rf_holdout': {'cards': holdout, 'mae': rf_mae, 'rmse': rf_rmse},
    'curve_holdout_mae': curve_mae_ho,
}})

bucket = get_bucket()
upload_file(bucket, 'models/game_fps_model.csv', table_path, 'text/csv')
upload_json(bucket, 'models/fps_curve.json', curve_meta)
print('uploaded models/game_fps_model.csv + models/fps_curve.json')
